# Marked Power Spectrum estimator example

This notebook demonstrates how to use the Marked Power Spectrum Statistics estimator from the ACM package.

This estimator is wrapped around `jaxpower` and requires the `JaxpowerBackend` provided in `acm.estimators.galaxy_clustering.backends.jaxpower`.

The mark is defined as a polynomial of the smoothed density contrast,

$$m(\mathbf{x}) = c_0 + c_1\,\delta_R(\mathbf{x}) + c_2\,\delta_R(\mathbf{x})^2 + \cdots,$$

so the density contrast must be set before calling `set_mark`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.marked_spectrum import MarkedPowerSpectrumMultipoles

setup_logging()

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

# Instantiate class
estimator = MarkedPowerSpectrumMultipoles(
    backend="jaxpower",
    data_positions=data_positions,
    boxsize=boxsize,
    cellsize=5.0,
)

# Set the smoothed density contrast field used to construct the mark
estimator.backend.set_density_contrast(
    smoothing_radius=25.0,  # Mpc/h
    resampler="tsc",
    interlacing=3,
    compensate=True,
)

# Set the polynomial mark. Here m(x) = 1 - delta_R(x).
estimator.set_mark(coefficients=(1.0, -1.0))

# Compute the marked power spectrum multipoles with specified parameters
result = estimator.compute(
    edges={"step": 0.001},
    ells=(0, 2, 4),
    los=los,
    resampler="tsc",
    interlacing=3,
    compensate=True,
)

# Use the helper function to plot the result
fig, ax = estimator.plot(result)
ax.legend()
ax.grid(True, alpha=0.3)

*Note: `set_density_contrast` must be called before `set_mark` unless the backend already contains a density-contrast field. The same `resampler` should be used when constructing and computing the marked field. Not passing `resampler`, `interlacing` and `compensate` will use their default values in the underlying routines.*

In [ ]:
# Access the data from the result object

fig, ax = plt.subplots(figsize=(8, 6))

binned_result = result.select(k=slice(0, None, 5))
k = binned_result.flatten(level=None)[0].coords("k")
for ell in (0, 2, 4):
    pole = binned_result.get(ells=ell)
    ax.plot(k, pole * k, label=rf"$\ell={ell}$", alpha=0.8)

ax.set_xlabel(r"$k$ [h/Mpc]")
ax.set_ylabel(r"$k M(k)$ [(Mpc/h)$^3$]")
ax.set_title("Marked Power Spectrum Multipoles")
ax.legend()
ax.grid(True, alpha=0.3)